# positive_control_check — did the control fit its own objective

The control fell on the judge at both scored checkpoints: Overall −0.387 at step 100 and −0.574 at
step 500, against +0.055 for rerunning the baseline unchanged on these same rows. Prompt following
held at step 100 (−0.063) while subject consistency collapsed (−0.636).

`ref_img_dropout_prob: 0.5` is not the cause. The dataset computes
`drop_ref_img = drop_prompt and random.random() < ref_img_dropout_prob`, gated on prompt dropout at
1e-4, so the effective rate is 5e-5 and under one row was affected across the whole run.

Training minimises velocity MSE against the target image. This notebook measures whether the
generated images moved toward those targets at all, which the judge's metrics do not report.

Two comparisons per generated image, both deterministic and independent of GPT-4.1: similarity to
the row's target, and similarity to the row's references. Instruments are DINOv2-large, CLIP-large
and ArcFace, the same set `arms_embed` ran on the test split, so the control's reference-side
numbers are directly comparable to the arms'.

Four runs: `baseline`, `baseline__rerun`, and the two control checkpoints. The rerun is an identical
configuration regenerated and rescored, so it supplies the measured noise floor for both measures on
these exact rows.

Outcomes are distinguishable. Target similarity rising with steps means the run fit its objective
and the judge scored something else. Target similarity falling means the run did not fit its own
objective at these checkpoints, and the scored grid starts too late: `b_norm` reached 117% of the
arm's entire 4,000-step norm by step 50, and the earliest checkpoint generated was step 100 at 223%.

T4. No API calls.

## 1. Setup

In [1]:
from google.colab import drive
import os, sys, json, subprocess

drive.mount("/content/drive")
DRIVE  = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
SCORES = f"{DRIVE}/scores"
IMGS, EXPECTED = "/content/imgs", 37063

def n_files(d): return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    import shutil
    assert shutil.disk_usage("/content").free / 1e9 > 55, "the extracted image set needs ~50 GB"
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{DRIVE}/imgs_subset.tar" /content/imgs_subset.tar && '
                   f'tar -xf /content/imgs_subset.tar -C {IMGS}', shell=True, check=True)
    assert n_files(IMGS) >= EXPECTED, f"restore incomplete: {n_files(IMGS)} of {EXPECTED}"
print("images:", n_files(IMGS))

if not os.path.exists("/content/screen.jsonl"):
    subprocess.run(["cp", f"{DRIVE}/screen.jsonl", "/content/screen.jsonl"], check=True)
rows = [json.loads(l) for l in open("/content/screen.jsonl", encoding="utf-8")]
assert len(rows) == 302, f"expected 302 screen rows, got {len(rows)}"

import torch, torch.nn.functional as F, numpy as np
from PIL import Image
from transformers import AutoImageProcessor, AutoModel, CLIPImageProcessor, CLIPVisionModelWithProjection

device = "cuda" if torch.cuda.is_available() else "cpu"
dino_proc = AutoImageProcessor.from_pretrained("facebook/dinov2-large")
dino      = AutoModel.from_pretrained("facebook/dinov2-large").to(device).eval()
clip_proc = CLIPImageProcessor.from_pretrained("openai/clip-vit-large-patch14")
clip      = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14").to(device).eval()

@torch.no_grad()
def dino_emb(imgs):
    return F.normalize(dino(**dino_proc(images=imgs, return_tensors="pt").to(device)).pooler_output.float(), dim=-1)

@torch.no_grad()
def clip_emb(imgs):
    return F.normalize(clip(**clip_proc(images=imgs, return_tensors="pt").to(device)).image_embeds.float(), dim=-1)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface", "onnxruntime-gpu"], check=False)

# insightface's own downloader does not retry and resolves a different release tag depending on its
# version; arms_embed pinned v0.7, so the same build is fetched here and the numbers stay comparable
IF_DIR = os.path.expanduser("~/.insightface/models")
if not os.path.isdir(f"{IF_DIR}/buffalo_l"):
    os.makedirs(IF_DIR, exist_ok=True)
    subprocess.run(["rm", "-rf", f"{IF_DIR}/buffalo_l", f"{IF_DIR}/buffalo_l.zip"], check=False)
    subprocess.run(["curl", "-L", "--retry", "5", "--retry-all-errors", "--fail",
                    "-o", f"{IF_DIR}/buffalo_l.zip",
                    "https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_l.zip"],
                   check=True)
    subprocess.run(["unzip", "-q", "-o", f"{IF_DIR}/buffalo_l.zip", "-d", f"{IF_DIR}/buffalo_l"], check=True)

from insightface.app import FaceAnalysis
app = FaceAnalysis(name="buffalo_l", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

def to_bgr(pil):
    return np.array(pil.convert("RGB"))[:, :, ::-1]

def largest(faces):
    return max(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1])) if faces else None

print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "cpu only")
print("insightface providers:", app.models["recognition"].session.get_providers())

Mounted at /content/drive
images: 37063


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/549 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.71GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

[transformers] CLIPVisionModelWithProjection LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{

device: cuda | Tesla T4
insightface providers: ['CPUExecutionProvider']


## 2. Runs and target/reference cache

Both the 62 targets and the 223 references are embedded once and shared across the four runs, so
the per-run cost is 302 generated images. `baseline` names its files `test_{i:05d}.png` and the
other runs `{i:05d}.png`; both forms are resolved by probing.

In [2]:
RUNS = [
    ("baseline",         "baseline",                     "screen__baseline"),
    ("baseline__rerun",  "baseline__rerun",              "screen__baseline__rerun"),
    ("step100",          "positive_control_r8_step100",  "screen__positive_control_r8_step100"),
    ("step500",          "positive_control_r8_step500",  "screen__positive_control_r8_step500"),
]

def gen_path(gdir, i):
    for pat in (f"{i:05d}.png", f"test_{i:05d}.png"):
        p = f"{DRIVE}/gen/screen/{gdir}/{pat}"
        if os.path.exists(p): return p
    return None

print(f"{'run':18s} {'gen dir':32s} {'pngs':>5s} {'pattern':>16s} {'scores':>8s}")
for lab, gdir, sfile in RUNS:
    d = f"{DRIVE}/gen/screen/{gdir}"
    n = len([f for f in os.listdir(d) if f.endswith(".png")]) if os.path.isdir(d) else 0
    p0 = gen_path(gdir, 0)
    pat = os.path.basename(p0).replace("00000", "{i:05d}") if p0 else "MISSING"
    ok = os.path.exists(f"{SCORES}/{sfile}.jsonl")
    print(f"{lab:18s} {gdir:32s} {n:>5d} {pat:>16s} {'yes' if ok else 'MISSING':>8s}")
    assert n >= 302 and p0 and ok, f"{lab}: incomplete"

uid = lambda r: r["output_image"] + "|" + "|".join(sorted(r["input_images"]))
UID = {i: uid(r) for i, r in enumerate(rows)}
TGT = {i: os.path.join(IMGS, r["output_image"]) for i, r in enumerate(rows)}
REF = {i: [os.path.join(IMGS, p) for p in r["input_images"]] for i, r in enumerate(rows)}
uniq = sorted(set(TGT.values()) | {p for v in REF.values() for p in v})

import time
t0 = time.time()
if "EMB" not in dir() or len(EMB) != len(uniq):
    EMB = {}
for k, p in enumerate(uniq if len(EMB) != len(uniq) else []):
    im = Image.open(p).convert("RGB")
    fl = largest(app.get(to_bgr(im)))
    EMB[p] = (dino_emb([im])[0], clip_emb([im])[0], None if fl is None else fl.normed_embedding)
    if (k + 1) % 100 == 0: print(f"  {k+1}/{len(uniq)}  {time.time()-t0:.0f}s", flush=True)
print(f"cache: {len(EMB)} images ({len(set(TGT.values()))} targets, "
      f"{len({p for v in REF.values() for p in v})} references) in {time.time()-t0:.0f}s")

run                gen dir                           pngs          pattern   scores
baseline           baseline                           302      {i:05d}.png      yes
baseline__rerun    baseline__rerun                    302      {i:05d}.png      yes
step100            positive_control_r8_step100        302      {i:05d}.png      yes
step500            positive_control_r8_step500        302      {i:05d}.png      yes
  100/285  143s
  200/285  245s
cache: 285 images (62 targets, 223 references) in 324s


## 3. Similarity to target and to references

One file per run, `screen__<run>_similarity.jsonl`. Save-as-you-go with skip-if-exists on the row
index, so a dropped session resumes.

In [4]:
def row_sims(i, gpath):
    im = Image.open(gpath).convert("RGB")
    gd, gc = dino_emb([im])[0], clip_emb([im])[0]
    gen_embs = [f.normed_embedding for f in app.get(to_bgr(im))]
    td, tc, tf = EMB[TGT[i]]
    refs = REF[i]
    # arcface: each source face best-matched to any generated face, averaged over sources with a face.
    # None where no source or no generated image carries one, as in baseline_arcface.
    def face_sim(sources):
        if not gen_embs:
            return None
        s = [max(float(EMB[p][2] @ ge) for ge in gen_embs) for p in sources if EMB[p][2] is not None]
        return sum(s) / len(s) if s else None
    return dict(dino_tgt=float(gd @ td), clip_tgt=float(gc @ tc),
                arc_tgt=(max(float(tf @ ge) for ge in gen_embs) if tf is not None and gen_embs else None),
                dino_ref=float(torch.stack([gd @ EMB[p][0] for p in refs]).mean()),
                clip_ref=float(torch.stack([gc @ EMB[p][1] for p in refs]).mean()),
                arc_ref=face_sim(refs),
                gen_faces=len(gen_embs))

META = {r["i"]: r for r in (json.loads(l) for l in open(f"{SCORES}/screen__baseline.jsonl", encoding="utf-8"))}
assert all(META[i]["uid"] == UID[i] for i in META), "score-file uid does not match screen.jsonl at the same i"

for lab, gdir, _ in RUNS:
    out = f"{SCORES}/screen__{lab}_similarity.jsonl"
    done = {json.loads(l)["i"] for l in open(out, encoding="utf-8")} if os.path.exists(out) else set()
    todo = [i for i in range(len(rows)) if i not in done]
    print(f"\n{lab}: {len(done)} done, {len(todo)} to measure", flush=True)
    t0 = time.time()
    with open(out, "a", encoding="utf-8") as f:
        for k, i in enumerate(todo):
            f.write(json.dumps({"i": i, "uid": UID[i], "n_refs": META[i]["n_refs"],
                                "ref_kind": META[i]["ref_kind"], **row_sims(i, gen_path(gdir, i))}) + "\n")
            if (k + 1) % 100 == 0:
                f.flush(); print(f"  {k+1}/{len(todo)}  {time.time()-t0:.0f}s", flush=True)
    n = sum(1 for _ in open(out, encoding="utf-8"))
    assert n == len(rows), f"{lab}: {n} rows, expected {len(rows)}"
    print(f"  {lab}: {n} rows in {time.time()-t0:.0f}s")


baseline: 54 done, 248 to measure
  100/248  166s
  200/248  321s
  baseline: 302 rows in 389s

baseline__rerun: 0 done, 302 to measure
  100/302  148s
  200/302  289s
  300/302  432s
  baseline__rerun: 302 rows in 433s

step100: 0 done, 302 to measure
  100/302  151s
  200/302  294s
  300/302  444s
  step100: 302 rows in 447s

step500: 0 done, 302 to measure
  100/302  155s
  200/302  295s
  300/302  444s
  step500: 302 rows in 446s


## 4. Curve against the judge

Each run's similarity to target and to references, beside the GPT-4.1 scores already on file.
`baseline__rerun` is an identical configuration, so its row gives the size of a null move on both
measures.

In [5]:
import math, statistics as st
from scipy import stats

ov = lambda r: math.sqrt(max(r["PF"], 0) * max(r["SC"], 0))
load_sim = lambda lab: {json.loads(l)["i"]: json.loads(l)
                        for l in open(f"{SCORES}/screen__{lab}_similarity.jsonl", encoding="utf-8")}
def load_gpt(sfile):
    d = {}
    for l in open(f"{SCORES}/{sfile}.jsonl", encoding="utf-8"):
        r = json.loads(l)
        if "error" not in r: d[r["i"]] = r
    return d

SIM, GPT = {}, {}
for lab, _, sfile in RUNS:
    SIM[lab], GPT[lab] = load_sim(lab), load_gpt(sfile)
I = sorted(set.intersection(*[set(SIM[l]) for l, _, _ in RUNS], *[set(GPT[l]) for l, _, _ in RUNS]))
print(f"rows common to all runs: {len(I)}\n")

KEYS = ("dino_tgt", "clip_tgt", "arc_tgt", "dino_ref", "clip_ref", "arc_ref")
mean_ok = lambda s, k: st.mean([s[i][k] for i in I if s[i][k] is not None]) if any(s[i][k] is not None for i in I) else float("nan")

print(f"{'run':18s} " + " ".join(f"{k:>9s}" for k in KEYS) + f" {'PF':>7s} {'SC':>7s} {'Overall':>8s}")
for lab, _, _ in RUNS:
    s, g = SIM[lab], GPT[lab]
    print(f"{lab:18s} "
          + " ".join(f"{mean_ok(s, k):>9.4f}" for k in KEYS)
          + f" {st.mean([g[i]['PF'] for i in I]):>7.3f} {st.mean([g[i]['SC'] for i in I]):>7.3f}"
          + f" {st.mean([ov(g[i]) for i in I]):>8.3f}")

print(f"\npaired delta against baseline (n={len(I)})")
print(f"{'run':18s} {'measure':10s} {'delta':>9s} {'SE':>8s} {'p':>10s}")
for lab, _, _ in RUNS:
    if lab == "baseline": continue
    for k in KEYS:
        u = [i for i in I if SIM[lab][i][k] is not None and SIM["baseline"][i][k] is not None]
        if len(u) < 10:
            print(f"{lab:18s} {k:10s} {len(u):>9d}  too few paired rows"); continue
        d = np.array([SIM[lab][i][k] - SIM["baseline"][i][k] for i in u], float)
        se = d.std(ddof=1) / math.sqrt(len(d))
        print(f"{lab:18s} {k:10s} {d.mean():>+9.4f} {se:>8.4f} {stats.ttest_1samp(d, 0)[1]:>10.3g}  n={len(u)}")
    d = np.array([ov(GPT[lab][i]) - ov(GPT["baseline"][i]) for i in I], float)
    se = d.std(ddof=1) / math.sqrt(len(d))
    print(f"{lab:18s} {'GPT Overall':10s} {d.mean():>+9.4f} {se:>8.4f} {stats.ttest_1samp(d, 0)[1]:>10.3g}\n")

rows common to all runs: 302

run                 dino_tgt  clip_tgt   arc_tgt  dino_ref  clip_ref   arc_ref      PF      SC  Overall
baseline              0.8197    0.9042    0.2964    0.3952    0.7487    0.2955   8.017   8.079    7.955
baseline__rerun       0.8194    0.9049    0.2963    0.3952    0.7493    0.2959   8.093   8.106    8.010
step100               0.8486    0.9192    0.3770    0.3769    0.7351    0.2493   7.954   7.444    7.568
step500               0.8620    0.9174    0.4101    0.3785    0.7291    0.2784   7.791   7.258    7.381

paired delta against baseline (n=302)
run                measure        delta       SE          p
baseline__rerun    dino_tgt     -0.0002   0.0006       0.69  n=302
baseline__rerun    clip_tgt     +0.0006   0.0005      0.258  n=302
baseline__rerun    arc_tgt      -0.0001   0.0013      0.925  n=252
baseline__rerun    dino_ref     -0.0001   0.0004      0.886  n=302
baseline__rerun    clip_ref     +0.0006   0.0004      0.176  n=302
baseline__rerun 

## 5. Row level, and verdict

Whether the rows the training moved toward their targets are the rows the judge scored up. A
positive correlation means the two agree about what improved; near zero means the judge is
scoring something the training objective does not control.

In [6]:
print("row-level correlation of deltas against baseline (n=%d):" % len(I))
print(f"{'run':18s} {'x':10s} {'vs':14s} {'Pearson':>9s} {'Spearman':>9s} {'p':>10s}")
for lab, _, _ in RUNS:
    if lab == "baseline": continue
    y_sc = np.array([GPT[lab][i]["SC"] - GPT["baseline"][i]["SC"] for i in I], float)
    y_ov = np.array([ov(GPT[lab][i]) - ov(GPT["baseline"][i]) for i in I], float)
    for k in ("dino_tgt", "dino_ref", "arc_ref"):
        u = [i for i in I if SIM[lab][i][k] is not None and SIM["baseline"][i][k] is not None]
        if len(u) < 10: continue
        x = np.array([SIM[lab][i][k] - SIM["baseline"][i][k] for i in u], float)
        y_sc_u = np.array([GPT[lab][i]["SC"] - GPT["baseline"][i]["SC"] for i in u], float)
        y_ov_u = np.array([ov(GPT[lab][i]) - ov(GPT["baseline"][i]) for i in u], float)
        for yl, y in (("GPT SC", y_sc_u), ("GPT Overall", y_ov_u)):
            pr, sp = stats.pearsonr(x, y), stats.spearmanr(x, y)
            print(f"{lab:18s} {k:10s} {yl:14s} {pr[0]:>+9.3f} {sp[0]:>+9.3f} {pr[1]:>10.3g}")
    print()

null = {}
for k in KEYS:
    u = [i for i in I if SIM["baseline__rerun"][i][k] is not None and SIM["baseline"][i][k] is not None]
    null[k] = st.mean([SIM["baseline__rerun"][i][k] - SIM["baseline"][i][k] for i in u]) if u else float("nan")
print("null move, identical configuration regenerated:",
      " ".join(f"{k} {v:+.4f}" for k, v in null.items()))

d100 = st.mean([SIM["step100"][i]["dino_tgt"] - SIM["baseline"][i]["dino_tgt"] for i in I])
d500 = st.mean([SIM["step500"][i]["dino_tgt"] - SIM["baseline"][i]["dino_tgt"] for i in I])
print(f"\ntarget similarity, DINOv2: step100 {d100:+.4f}, step500 {d500:+.4f}, "
      f"null {null['dino_tgt']:+.4f}")

print("\nverdict:")
if d100 > abs(null["dino_tgt"]) and d500 > d100:
    print("  The run fit its objective: generated images moved toward their targets, and further")
    print("  training moved them further. The judge's fall is therefore not a failure to train, and")
    print("  what the metric rewards is the thing to examine rather than the recipe.")
elif d500 < d100 < abs(null["dino_tgt"]):
    print("  The run did not fit its objective at either scored checkpoint, and moved away from the")
    print("  targets as training continued. Both checkpoints sit past the arm's entire trajectory:")
    print("  b_norm reached 117% of the arm's step-4000 norm by step 50, and step 100 is at 223%.")
    print("  A finer grid below step 100 is the next thing to generate, not a different recipe.")
else:
    print("  Mixed: the two checkpoints disagree in direction, or the move is inside the null.")
    print("  Read the per-run table above before choosing the next run.")

row-level correlation of deltas against baseline (n=302):
run                x          vs               Pearson  Spearman          p
baseline__rerun    dino_tgt   GPT SC            +0.020    +0.073      0.725
baseline__rerun    dino_tgt   GPT Overall       +0.001    +0.016      0.984
baseline__rerun    dino_ref   GPT SC            -0.014    -0.041      0.813
baseline__rerun    dino_ref   GPT Overall       -0.040    -0.059      0.485
baseline__rerun    arc_ref    GPT SC            -0.035    -0.112      0.618
baseline__rerun    arc_ref    GPT Overall       -0.044    -0.105       0.53

step100            dino_tgt   GPT SC            +0.090    +0.062      0.121
step100            dino_tgt   GPT Overall       +0.115    +0.108     0.0455
step100            dino_ref   GPT SC            +0.019    +0.074      0.743
step100            dino_ref   GPT Overall       -0.079    +0.041      0.171
step100            arc_ref    GPT SC            -0.016    +0.091      0.823
step100            arc_ref   